# Scattering transforms for radio signals

A scattering transform is a fixed wavelet network: convolve with complex wavelets, apply modulus, and average. First-order coefficients describe energy in wavelet bands. Second-order coefficients describe how that band energy changes over time.

For IQ signal $x$, $S_1x=|x*\psi_{\lambda_1}|*\phi_T$ and $S_2x=||x*\psi_{\lambda_1}|*\psi_{\lambda_2}|*\phi_T$.

The representation is stable to small time deformations and becomes invariant to translation and global carrier phase. This is useful in modulation recognition when phase and arrival time are nuisance variables. This notebook uses a compact educational PyTorch implementation; use a tested package such as Kymatio for production.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchsig_models.utils.training import set_deterministic

set_deterministic(7)
N = 1024
FREQUENCIES = torch.logspace(np.log10(0.015), np.log10(0.4), 12)


## Morlet filters and scattering coefficients

Morlet wavelets are localized complex sinusoids. Log-spaced center frequencies provide roughly constant relative bandwidth.

In [ ]:
def morlet_bank(length, frequencies, cycles=6.0):
    time = torch.arange(length) - length // 2
    filters = []
    for frequency in frequencies:
        sigma = cycles / (2 * torch.pi * frequency)
        wavelet = torch.exp(-0.5 * (time / sigma).square())
        wavelet = wavelet * torch.exp(2j * torch.pi * frequency * time)
        wavelet = wavelet - wavelet.mean()
        filters.append(wavelet / wavelet.abs().square().sum().sqrt())
    return torch.stack(filters)

def wavelet_modulus(signals, wavelets):
    spectrum = torch.fft.fft(signals, dim=-1)
    filters = torch.fft.fft(torch.fft.ifftshift(wavelets, dim=-1), dim=-1)
    return torch.fft.ifft(spectrum.unsqueeze(-2) * filters, dim=-1).abs()

def scattering_features(signals, wavelets):
    if signals.ndim == 1:
        signals = signals.unsqueeze(0)
    first = wavelet_modulus(signals, wavelets)
    features = [signals.abs().mean(-1, keepdim=True), first.mean(-1)]
    second = []
    for j in range(1, wavelets.shape[0]):
        envelopes = wavelet_modulus(first[:, j], wavelets)
        second.append(envelopes[:, :j].mean(-1))
    features.append(torch.cat(second, dim=-1))
    return torch.cat(features, dim=-1)

WAVELETS = morlet_bank(N, FREQUENCIES)
responses = torch.fft.fftshift(torch.fft.fft(torch.fft.ifftshift(WAVELETS, dim=-1)), dim=-1)
for response in responses:
    plt.plot(torch.fft.fftshift(torch.fft.fftfreq(N)), response.abs())
plt.xlim(0, .5); plt.title("Morlet filter bank"); plt.xlabel("Normalized frequency")
plt.show()


## Modulated signal and scalogram

The unaveraged first layer is a constant-Q-like scalogram. Its ridges expose carrier bands, bursts, and envelope dynamics.

In [ ]:
phases = torch.randint(0, 4, (N // 8,)) * torch.pi / 2
signal = torch.exp(1j * phases).repeat_interleave(8)
signal += 0.15 * (torch.randn(N) + 1j * torch.randn(N))
scalogram = wavelet_modulus(signal, WAVELETS).squeeze(0)

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(signal.real[:250], label="I"); axes[0].plot(signal.imag[:250], label="Q")
axes[0].legend(); axes[0].set_title("Noisy QPSK")
axes[1].imshow(scalogram, origin="lower", aspect="auto", cmap="magma")
axes[1].set_title("First-order wavelet modulus")
plt.tight_layout()


## Phase and shift invariance

A phase rotation only rotates complex wavelet responses; modulus removes it. Global averaging removes circular time location.

In [ ]:
reference = scattering_features(signal, WAVELETS)
rotated = scattering_features(signal * torch.exp(torch.tensor(1.3j)), WAVELETS)
shifted = scattering_features(torch.roll(signal, 37), WAVELETS)

def relative_change(a, b):
    return ((a-b).norm() / a.norm().clamp_min(1e-8)).item()

print("Scattering change, phase rotation:", f"{relative_change(reference, rotated):.3e}")
print("Scattering change, circular shift:", f"{relative_change(reference, shifted):.3e}")
print("Feature dimension:", reference.shape[-1])


## Applicability to modrec

Scattering is attractive with limited labels, strong phase/timing nuisance variation, or tight latency budgets. Local rather than global averaging preserves symbol and burst timing. Limitations include loss of absolute phase information, fixed filters that may not match every waveform, and weaker long-range modeling than learned sequence architectures.

Compare accuracy and macro F1 across SNR, robustness to phase/frequency/timing offsets, labeled-data efficiency, latency, and feature size.